# Fix WCS in  reduced MEF file, run sextractor and scamp.

This script is used after the pymefpp_solve_field script. It does the following:

* copies the <root>_pre.fits MEF file renaming it to <root>_pre_wcs.fits
* go extension by extension:
    * read the corresponding solve-field wcs file from the extension directory
    * correct the WCS headers of the <root>_pre_wcs.fits 
    * creates the <root>_pre_wcs.ahead file for scamp
* run sextractor on <root>_pre_wcs.fits and creates the <root>_pre_wcs.cat file for scamp
* run scamp on <root>_pre_wcs.cat using <root>_pre_wcs.ahead and creates <root>_pre_wcs.head
    
At the end it writes a wcs file with the fully correct WCS information to be input to swarp

In [ ]:
import os
import glob
from textwrap import wrap
from astropy.io import fits
import numpy as np
import subprocess
import builtins
import re
import copy
import types


cwd = os.getcwd()
print('cwd: ', cwd)

In [ ]:
# ======================================================================
# A utility function
# ======================================================================
def runCmd( str_cmd, p_shell=True ):
    """ 
    A wrapper to run system commands.
      
    Parameters
    ----------
    str_cmd: str      
      Command string to be executed in the shell
    
    p_shell: bool
      If True (default), command will be executed through the shell, and all 
      cout/cerr messages will be available.
      If False, exception is the only way to find out problems during the call.

    Returns
    -------
      0 if some error, or 1 if all was OK

    Notes
    -----
    TODO:
      - allow to launch commands in background 
      - best checking of error when shell=True
    """
           
    print("log.info: Running command: %s \n"%(str_cmd,))
    
    try:
        p = subprocess.Popen(str_cmd, bufsize = 0, shell = p_shell, stdin = subprocess.PIPE, 
                             stdout = subprocess.PIPE, stderr = subprocess.PIPE, 
                             close_fds = True)
    except:
        print("Some error while running command...")
        raise
       
    #Warning
    #We use communicate() rather than .stdin.write, .stdout.read or .stderr.read 
    #to avoid deadlocks due to any of the other OS pipe buffers filling up and 
    #blocking the child process.(Python Ref.doc)

    (stdoutdata, stderrdata) = p.communicate()
    err = stdoutdata.decode('ascii') + " " + stderrdata.decode('ascii')

    # IMPORTANT: Next checking (only available when shell=True) not always detect all kind of errors !!
    if err.count('WARNING: Significant inaccuracy'):
        print("Canno't get accuracy astrometric calibration")
        return 2
    elif (err.count('ERROR ') or err.count('error ') or err.count("Error ")\
      or err.count('*Error*')\
      or err.count('Segmentation fault') or err.count("command not found") \
      or err.count('No source found') \
      or err.count("No such file or directory")
      or err.count('WARNING: Not enough matched detections') # SCAMP
      or err.count('WARNING: Significant inaccuracy likely to occur in projection')): #SCAMP
        print("An error happened while running command --> %s \n"%(err,))
        print(stderrdata.decode('ascii'))
        return 1 # ERROR
    else:
        print(stdoutdata.decode('ascii'))
        return 0 # NO ERROR
    


In [ ]:
class extWCS(object):
    """Very simple WCS class to contain WCS solution for
    each extension headers of a MEF file."""
    def __init__(self, extension_name, crval1, crval2, crpix1, crpix2, cd1_1, cd1_2, cd2_1, cd2_2):
        self.name = extension_name
        self.crval1 = crval1
        self.crval2 = crval2
        self.crpix1 = crpix1
        self.crpix2 = crpix2
        self.cd1_1 = cd1_1
        self.cd1_2 = cd1_2
        self.cd2_1 = cd2_1
        self.cd2_2 = cd2_2
        self.theta =  np.arccos(cd1_1)

In [ ]:
def create_wcs_frame(file_name):
    """Creates the <root>_pre_wcs.fits file from input 
    <root>_pre.fits, the output of mefpp, by adding the
    WCS solutions found by solve-field for each extension.
    The file is written into the directory <root>_pre"""
    dir_name = os.path.abspath(file_name[:-5])

    if os.path.isfile(file_name):
        print('file exists: ', file_name)
    if os.path.isdir(dir_name):
        print('directory exists: ', dir_name)
    
    hdu_list = fits.open(file_name)
    ext_wcs = {}
    for an_hdu in  hdu_list[1:]:
        #list_of_extensions.append(an_hdu.header['EXTNAME'])
        ext_file_name = an_hdu.header['EXTNAME'].replace('#','_')
        solve_field_dir_name = dir_name+'/'+ext_file_name+'_solve_field/'
        if os.path.isdir(solve_field_dir_name):
            print('solve field directory exists: ', solve_field_dir_name)
        else:
            print('solve field directory does not exists: ', solve_field_dir_name)
            break
            
        wcs_file = solve_field_dir_name+'/'+ext_file_name+'.wcs'
        if os.path.isfile(wcs_file):
            print('WCS file exists: ', wcs_file)
        else:
            print('**WCS file does not exists: ', wcs_file)
            break
        #print(get_wcs_fromfile(wcs_file))
        ext_wcs[an_hdu.header['EXTNAME']] = get_wcs_fromfile(wcs_file, an_hdu.header['EXTNAME'])


    # here we loop over the ext_wcs dictionary to print the WCS keys
    zkeys = ext_wcs.keys()
    print(len(zkeys))

    for akey in ext_wcs.keys():
        print('----------> ', akey, ' <----------')
        #print('crval1: ', ext_wcs[akey].crval1)
        #print('crval2: ', ext_wcs[akey].crval2)
        #print('crpix1: ', ext_wcs[akey].crpix1)
        #print('crpix2: ', ext_wcs[akey].crpix2)
        #print('cd1_1: ', ext_wcs[akey].cd1_1)
        #print('cd1_2: ', ext_wcs[akey].cd1_2)
        #print('cd2_1: ', ext_wcs[akey].cd2_1)
        #print('cd2_2: ', ext_wcs[akey].cd2_2)
        print('theta: ', 360.*ext_wcs[akey].theta/(2.*np.pi))

    # here we are ready to create the modified MEF file with the corrected headers
    for an_hdu in  hdu_list[1:]:
        #list_of_extensions.append(an_hdu.header['EXTNAME'])
        header = an_hdu.header
        header_name = header['EXTNAME']
        header['CRVAL1'] = ext_wcs[header_name].crval1
        header['CRVAL2'] = ext_wcs[header_name].crval2
        header['CRPIX1'] = ext_wcs[header_name].crpix1
        header['CRPIX2'] = ext_wcs[header_name].crpix2
        header['CD1_1'] = ext_wcs[header_name].cd1_1
        header['CD1_2'] = ext_wcs[header_name].cd1_2
        header['CD2_1'] = ext_wcs[header_name].cd2_1
        header['CD2_2'] = ext_wcs[header_name].cd2_2

    print('creating %s'%(dir_name+'/'+file_name[:-5]+'_wcs.fits',))
    hdu_list.writeto(dir_name+'/'+file_name[:-5]+'_wcs.fits', output_verify='ignore', overwrite=True)    
    hdu_list.close()

In [ ]:
def get_wcs_fromfile(wcs_file, ext_name):
    """This script reads the wcs file output from solve-field
    and returns a WCS structure with the WCS keywords receiving
    their values from the wcs file."""
    f = open(wcs_file, 'r')
    wcs_lines = f.readlines()
    f.close()
    for aline in wrap(wcs_lines[0], 80):
        field = aline.split()
        #print(aline)
        if 'CRVAL1' in aline:
            crval1 = float(field[2])
            #print('CRVAL1: %f'%(crval1,))
        if 'CRVAL2' in aline:
            crval2 = float(field[2])
            #print('CRVAL2: %f'%(crval2,))
        if 'CRPIX1' in aline:
            crpix1 = float(field[2])
            #print('CRPIX1: %i'%(crpix1,))
        if 'CRPIX2' in aline:
            crpix2 = float(field[2])
            #print('CRPIX2: %i'%(crpix2,))
        if 'CD1_1' in aline:
            cd1_1 = float(field[2])
            #print('CD1_1: %e'%(cd1_1,))
        if 'CD1_2' in aline:
            cd1_2 = float(field[2])
            #print('CD1_2: %e'%(cd1_2,))
        if 'CD2_1' in aline:
            cd2_1 = float(field[2])
            #print('CD2_1: %e'%(cd2_1,))
        if 'CD2_2' in aline:
            cd2_2 = float(field[2])
            #print('CD2_2: %e'%(cd2_2,))
    thisWCS = extWCS(ext_name, crval1, crval2, crpix1, crpix2, cd1_1, cd1_2, cd2_1, cd2_2)
    return thisWCS

In [ ]:
def write_sex_default_outpars(output_dir):
    """Minimal procedure to write a sextractor output
    pars file needed to subsequently run scamp."""
    lines = """NUMBER                   Running object number                                    
EXT_NUMBER               FITS extension number                                    
FLUX_ISO               #  Isophotal flux                                            [count]
FLUXERR_ISO            #  RMS error for isophotal flux                              [count]
MAG_ISO                #  Isophotal magnitude                                       [mag]
MAGERR_ISO             #  RMS error for isophotal magnitude                         [mag]
FLUX_APER              #  Flux vector within fixed circular aperture(s)             [count]
FLUXERR_APER           #  RMS error vector for aperture flux(es)                    [count]
MAG_APER               #  Fixed aperture magnitude vector                           [mag]
MAGERR_APER            #  RMS error vector for fixed aperture mag.                  [mag]
FLUX_AUTO              ##  Flux within a Kron-like elliptical aperture               [count]
FLUXERR_AUTO           #  RMS error for AUTO flux                                   [count]
MAG_AUTO               #  Kron-like elliptical aperture magnitude                   [mag]
MAGERR_AUTO            #  RMS error for AUTO magnitude                              [mag]
A_IMAGE                  Profile RMS along major axis                              [pixel]
B_IMAGE                  Profile RMS along minor axis                              [pixel]
THETA_IMAGE              Position angle (CCW/x)                                    [deg]
XWIN_IMAGE               Windowed position estimate along x                        [pixel]
YWIN_IMAGE               Windowed position estimate along y                        [pixel]
ERRX2WIN_IMAGE           Variance of windowed pos along x                          [pixel**2]
ERRY2WIN_IMAGE           Variance of windowed pos along y                          [pixel**2]
ERRXYWIN_IMAGE           Covariance of windowed pos between x and y                [pixel**2]
ERRX2WIN_WORLD           Variance of windowed pos along X-WORLD (alpha)            [deg**2]
ERRY2WIN_WORLD           Variance of windowed pos along Y-WORLD (delta)            [deg**2]
ERRXYWIN_WORLD           Covariance of windowed pos X-WORLD/Y-WORLD                [deg**2]
ERRAWIN_IMAGE            RMS windowed pos error along major axis                   [pixel]
ERRBWIN_IMAGE            RMS windowed pos error along minor axis                   [pixel]
ERRTHETAWIN_IMAGE        Windowed error ellipse pos angle (CCW/x)                  [deg]
FLAGS                    Extraction flags                                         
FLAGS_WEIGHT             Weighted extraction flags                                
FLAGS_WIN                Flags for WINdowed parameters                            
FWHM_IMAGE               FWHM assuming a gaussian core                             [pixel]
ELONGATION               A_IMAGE/B_IMAGE                                          
ELLIPTICITY              1 - B_IMAGE/A_IMAGE                                      
CLASS_STAR                S/G classifier output                                    
FLUX_RADIUS            #  Fraction-of-light radii                                   [pixel]"""
    open(output_dir+'/sex_default.param', 'w').close()
    with open(output_dir+'/sex_default.param', 'a') as f:
        for aline in lines.split('\n'):
            print(aline, file=f)

In [ ]:
def write_sex_default_nnw(output_dir):
    """Procedure to write basic sextractor .nnw file"""
    lines = """NNW
# Neural Network Weights for the SExtractor star/galaxy classifier (V1.3)
# inputs:	9 for profile parameters + 1 for seeing.
# outputs:	``Stellarity index'' (0.0 to 1.0)
# Seeing FWHM range: from 0.025 to 5.5'' (images must have 1.5 < FWHM < 5 pixels)
# Optimized for Moffat profiles with 2<= beta <= 4.

 3 10 10  1

-1.56604e+00 -2.48265e+00 -1.44564e+00 -1.24675e+00 -9.44913e-01 -5.22453e-01  4.61342e-02  8.31957e-01  2.15505e+00  2.64769e-01
 3.03477e+00  2.69561e+00  3.16188e+00  3.34497e+00  3.51885e+00  3.65570e+00  3.74856e+00  3.84541e+00  4.22811e+00  3.27734e+00

-3.22480e-01 -2.12804e+00  6.50750e-01 -1.11242e+00 -1.40683e+00 -1.55944e+00 -1.84558e+00 -1.18946e-01  5.52395e-01 -4.36564e-01 -5.30052e+00
 4.62594e-01 -3.29127e+00  1.10950e+00 -6.01857e-01  1.29492e-01  1.42290e+00  2.90741e+00  2.44058e+00 -9.19118e-01  8.42851e-01 -4.69824e+00
-2.57424e+00  8.96469e-01  8.34775e-01  2.18845e+00  2.46526e+00  8.60878e-02 -6.88080e-01 -1.33623e-02  9.30403e-02  1.64942e+00 -1.01231e+00
 4.81041e+00  1.53747e+00 -1.12216e+00 -3.16008e+00 -1.67404e+00 -1.75767e+00 -1.29310e+00  5.59549e-01  8.08468e-01 -1.01592e-02 -7.54052e+00
 1.01933e+01 -2.09484e+01 -1.07426e+00  9.87912e-01  6.05210e-01 -6.04535e-02 -5.87826e-01 -7.94117e-01 -4.89190e-01 -8.12710e-02 -2.07067e+01
-5.31793e+00  7.94240e+00 -4.64165e+00 -4.37436e+00 -1.55417e+00  7.54368e-01  1.09608e+00  1.45967e+00  1.62946e+00 -1.01301e+00  1.13514e-01
 2.20336e-01  1.70056e+00 -5.20105e-01 -4.28330e-01  1.57258e-03 -3.36502e-01 -8.18568e-02 -7.16163e+00  8.23195e+00 -1.71561e-02 -1.13749e+01
 3.75075e+00  7.25399e+00 -1.75325e+00 -2.68814e+00 -3.71128e+00 -4.62933e+00 -2.13747e+00 -1.89186e-01  1.29122e+00 -7.49380e-01  6.71712e-01
-8.41923e-01  4.64997e+00  5.65808e-01 -3.08277e-01 -1.01687e+00  1.73127e-01 -8.92130e-01  1.89044e+00 -2.75543e-01 -7.72828e-01  5.36745e-01
-3.65598e+00  7.56997e+00 -3.76373e+00 -1.74542e+00 -1.37540e-01 -5.55400e-01 -1.59195e-01  1.27910e-01  1.91906e+00  1.42119e+00 -4.35502e+00

-1.70059e+00 -3.65695e+00  1.22367e+00 -5.74367e-01 -3.29571e+00  2.46316e+00  5.22353e+00  2.42038e+00  1.22919e+00 -9.22250e-01 -2.32028e+00


 0.00000e+00 
 1.00000e+00"""
    open(output_dir+'/default.nnw', 'w').close()
    with open(output_dir+'/default.nnw', 'a') as f:
        for aline in lines.split('\n'):
            print(aline, file=f)

In [ ]:
def write_sex_default_conv(output_dir):
    """Procedure to write basic sextractor .conv file."""
    lines = """CONV NORM
# 3x3 ``all-ground'' convolution mask with FWHM = 2 pixels.
1 2 1
2 4 2
1 2 1"""
    open(output_dir+'/default.conv', 'w').close()
    with open(output_dir+'/default.conv', 'a') as f:
        for aline in lines.split('\n'):
            print(aline, file=f)

In [ ]:
def write_sex_default(output_dir, output_cat):
    """Procedure to write default.sex file, minimally."""
    lines = """# Default configuration file for SExtractor 2.19.5
# EB 2014-03-19
#
 
#-------------------------------- Catalog ------------------------------------
""" + "CATALOG_NAME    "+output_cat+ " # name of the output catalog" + """
CATALOG_TYPE     FITS_LDAC     # NONE,ASCII,ASCII_HEAD, ASCII_SKYCAT,
                                # ASCII_VOTABLE, FITS_1.0 or FITS_LDAC
PARAMETERS_NAME  sex_default.param  # name of the file containing catalog contents
 
#------------------------------- Extraction ----------------------------------
 
DETECT_TYPE      CCD            # CCD (linear) or PHOTO (with gamma correction)
DETECT_MINAREA   9              # min. # of pixels above threshold
DETECT_THRESH    3.5            # <sigmas> or <threshold>,<ZP> in mag.arcsec-2
ANALYSIS_THRESH  3.5            # <sigmas> or <threshold>,<ZP> in mag.arcsec-2
 
FILTER           Y              # apply filter for detection (Y or N)?
FILTER_NAME      default.conv   # name of the file containing the filter
 
DEBLEND_NTHRESH  32             # Number of deblending sub-thresholds
DEBLEND_MINCONT  0.005          # Minimum contrast parameter for deblending
 
CLEAN            Y              # Clean spurious detections? (Y or N)?
CLEAN_PARAM      1.0            # Cleaning efficiency
 
MASK_TYPE        CORRECT        # type of detection MASKing: can be one of
                                # NONE, BLANK or CORRECT
 
#------------------------------ Photometry -----------------------------------
 
PHOT_APERTURES   5              # MAG_APER aperture diameter(s) in pixels
PHOT_AUTOPARAMS  2.5, 3.5       # MAG_AUTO parameters: <Kron_fact>,<min_radius>
PHOT_PETROPARAMS 2.0, 3.5       # MAG_PETRO parameters: <Petrosian_fact>,
                                # <min_radius>
 
SATUR_LEVEL      50000.0        # level (in ADUs) at which arises saturation
SATUR_KEY        SATURATE       # keyword for saturation level (in ADUs)
 
MAG_ZEROPOINT    25.0            # magnitude zero-point
MAG_GAMMA        4.0            # gamma of emulsion (for photographic scans)
GAIN             2.6            # detector gain in e-/ADU
GAIN_KEY         GAIN           # keyword for detector gain in e-/ADU
PIXEL_SCALE      0            # size of pixel in arcsec (0=use FITS WCS info)
 
#------------------------- Star/Galaxy Separation ----------------------------
 
SEEING_FWHM      1.2            # stellar FWHM in arcsec
STARNNW_NAME     default.nnw    # Neural-Network_Weight table filename
 
#------------------------------ Background -----------------------------------
 
BACK_SIZE        64             # Background mesh: <size> or <width>,<height>
BACK_FILTERSIZE  3              # Background filter: <size> or <width>,<height>
 
BACKPHOTO_TYPE   GLOBAL         # can be GLOBAL or LOCAL
 
#------------------------------ Check Image ----------------------------------
 
CHECKIMAGE_TYPE  APERTURES      # can be NONE, BACKGROUND, BACKGROUND_RMS,
                                # MINIBACKGROUND, MINIBACK_RMS, -BACKGROUND,
                                # FILTERED, OBJECTS, -OBJECTS, SEGMENTATION,
                                # or APERTURES
CHECKIMAGE_NAME  check.fits     # Filename for the check-image
 
#--------------------- Memory (change with caution!) -------------------------
 
MEMORY_OBJSTACK  3000           # number of objects in stack
MEMORY_PIXSTACK  300000         # number of pixels in stack
MEMORY_BUFSIZE   1024           # number of lines in buffer
 
#----------------------------- Miscellaneous ---------------------------------
 
VERBOSE_TYPE     NORMAL         # can be QUIET, NORMAL or FULL
HEADER_SUFFIX    .head          # Filename extension for additional headers
WRITE_XML        N              # Write XML file (Y/N)?
XML_NAME         sex.xml        # Filename for XML output
"""
    open(output_dir+'/default.sex', 'w').close()
    with open(output_dir+'/default.sex', 'a') as f:
        for aline in lines.split('\n'):
            print(aline, file=f)


In [ ]:
def run_sextractor(filename):
    # here we run sextractor...
    dir_name = os.path.abspath(filename[:-5])
    write_sex_default(dir_name, filename[:-5]+'_wcs.cat')
    write_sex_default_conv(dir_name)
    write_sex_default_outpars(dir_name)
    write_sex_default_nnw(dir_name)
    os.chdir(dir_name)

    logfile = filename[:-5]+'_sex.log'
    logoutput = open(logfile, 'w+')
    print('sex logs '+logfile)

    try:
        p = subprocess.Popen([
        '/usr/bin/sex',    
       filename[:-5]+'_wcs.fits',
        '-c', 'default.sex'],
        stdout=logoutput,
        stderr=logoutput,
        universal_newlines=True)
        output, error = p.communicate()
    except subprocess.CalledProcessError as e:
        print(e.output)
    os.chdir('..')


In [ ]:
def write_scamp_ahead_file(file_name):
    # here we write the extension headers to the .ahead file for scamp use
    dir_name = os.path.abspath(file_name[:-5])
    print('dir_name: ', dir_name)
    hdu_list = fits.open(dir_name+'/'+file_name[:-5]+'_wcs.fits')
    
    # before writing the headers we add a few keywords in 8-char format
    header0 = hdu_list[0].header
    
    # translate filter keyword
    try:
        zfilter = header0['HIERARCH ESO INS FILT1 NAME']
        print('pymefpp: FILTER: '+zfilter)
    except:
        zfilter = 'NOT_FOUND'
        print('pymefpp: failed to determine FILTER. Setting it to NOT_FOUND')
        
    try:
        header0.insert('LST', ('FILTER', zfilter, 'pymefpp: Filter used for observation'), after=True)
        print('pymefpp: Setting FILTER to '+zfilter)
    except:
        print('pymefpp: failed to insert keyword FILTER after keyword LST')
    
    # translate airmass keyword
    try:
        airmass_start = header0['HIERARCH ESO TEL AIRM START']
        airmass_end = header0['HIERARCH ESO TEL AIRM END']
        airmass_obs = (airmass_start+airmass_end)/2.0
    except:
        airmass_obs = 1.0
        print('pymefpp: failed to determine airmass. Setting it to 1.0')
    
    try:
        header0.insert('FILTER', ('AIRMASS', airmass_obs, 'pymefpp: Average airmass of frame'), after=True)
        print('pymefpp: Setting AIRMASS to %5.3f'%(airmass_obs,))
    except:
        print('pymefpp: failed to insert keyword AIRMASS after FILTER')


    open(dir_name+'/'+file_name[:-5]+'_wcs.ahead', 'w').close()
    f = open(dir_name+'/'+file_name[:-5]+'_wcs.ahead', 'a')
    with open(dir_name+'/'+file_name[:-5]+'_wcs.ahead', 'a') as f:
        for an_hdu in hdu_list[1:]:
            header = an_hdu.header
            for aline in wrap(repr(header), 80):
                print(aline, file=f)
            print('END', file=f)
    hdu_list.close()

In [ ]:
# adapted to python3 from the PAPI pipeline

# ======================================================================

__version__ = "0.1 (2019-08-13)"

# ======================================================================


class SCAMP_Exception(Exception):
    pass

class SCAMP_AccuracyException(Exception):
    pass
# ======================================================================

class SCAMP:
    """
    A wrapper class to transparently use SCAMP.

    """

    _SC_config = { 

#----------------------------- Field grouping ---------------------------------
    
        "FGROUP_RADIUS":
        {"comment": "Max dist (deg) between field groups",
         "value": 1.0},

#---------------------------- Reference catalogs ------------------------------
        
        "REF_SERVER":
        {"comment": "Internet addresses of catalog servers",
         "value": "cocat1.u-strasbg.fr"},
        
        "REF_PORT":
        {"comment": "Ports to connect to catalog servers",
         "value": 1660},
        
        "CDSCLIENT_EXEC":
        {"comment": 'CDSclient executable',
         "value": "aclient"},
        
        "ASTREF_CATALOG":
        {"comment": "NONE, FILE, USNO-A1, USNO-A2, USNO-B1, GSC-1.3, GSC-2.2, UCAC-1, UCAC-2,NOMAD-1, 2MASS, DENIS-3, SDSS-R3, SDSS-R5 or SDSS-R6 ",
         "value": "USNO-B1"},
        
        "ASTREF_BAND":
        {"comment": "Photom. band for astr.ref.magnitudes or DEFAULT, BLUEST, or REDDEST",
         "value": "DEFAULT"},
                
        "ASTREFMAG_LIMITS":
        {"comment": "Select magnitude range in ASTREF_BAND",
         "value": [-99.0,99.0]},
        
        "ASTREFCAT_NAME":
        {"comment": "Local astrometric reference catalogs",
         "value": "NONE"},
        
#        "ASTREFCENT_KEYS":
#        {"comment": "Local ref.cat.centroid parameters",
#         "value": "X_WORLD,Y_WORLD"},
        
#        "ASTREFERR_KEYS":
#        {"comment": 'Local ref.cat.error ellipse parameters',
#         "value": 'ERRA_WORLD, ERRB_WORLD, ERRTHETA_WORLD'},
        
#        "ASTREFMAG_KEY":
#        {"comment": "Local ref.cat.magnitude parameter",
#         "value": "MAG"},
        
        "SAVE_REFCATALOG":
        {"comment": "Save ref catalogs in FITS-LDAC format?",
         "value": "Y"},
                
        "REFOUT_CATPATH":
        {"comment": "Save path for reference catalogs",
         "value": "."},

#--------------------------- Merged output catalogs ---------------------------
       
        "MERGEDOUTCAT_NAME":
        {"comment": "Merged output catalog filename",
         "value": "merged.cat"},
        
        "MERGEDOUTCAT_TYPE":
        {"comment": "NONE, ASCII_HEAD, ASCII, FITS_LDAC",
         "value": 'NONE'},

#--------------------------- Fulloutput catalogs ---------------------------
       
        "FULLOUTCAT_NAME":
        {"comment": "Full output catalog filename",
         "value": "full.cat"},
        
        "FULLOUTCAT_TYPE":
        {"comment": "NONE, ASCII_HEAD, ASCII, FITS_LDAC",
         "value": 'NONE'},
                
#----------------------------- Pattern matching -------------------------------
        
        "MATCH":
        {"comment": "Do pattern-matching (Y/N) ?",
         "value": "Y"},
        
        "MATCH_NMAX":
        {"comment": 'Max.number of detections for MATCHing(0=auto)',
         "value": 0},
        
        "PIXSCALE_MAXERR":
        {"comment": "Max scale-factor uncertainty",
         "value": 1.10},
        
        "POSANGLE_MAXERR":
        {"comment": 'Max position-angle uncertainty (deg)',
         "value": 5.0},
        
        "POSITION_MAXERR":
        {"comment": "Max positional uncertainty (arcmin)",
         "value": 0.1},
        
        "MATCH_RESOL":
        {"comment": "Matching resolution (arcsec); 0=auto",
         "value": 0},
        
        "MOSAIC_TYPE":
        {"comment": "UNCHANGED, SAME_CRVAL, SHARE_PROJAXIS, FIX_FOCALPLANE or LOOSE",
         "value": "SHARE_PROJAXIS"},
        
        "MATCH_FLIPPED":
        {"comment": "Allow matching with flipped axes?",
         "value": "N"},
        
        "FIXFOCALPLANE_NMIN":
        {"comment": "Min number of dets for FIX_FOCALPLANE",
         "value": 1},

#---------------------------- Cross-identification ----------------------------
        
        "CROSSID_RADIUS":
        {"comment": "Cross-id initial radius (arcsec)",
         "value": 2.0},

#---------------------------- Astrometric solution ----------------------------
        
        "SOLVE_ASTROM":
        {"comment": "Compute astrometric solution (Y/N) ?",
         "value": "Y"},
                
        "PROJECTION_TYPE":
        {"comment": "SAME, TPV or TAN",
         "value": "SAME"},
        
        "ASTRINSTRU_KEY":
        {"comment": "FITS keyword(s) defining the astrom",
         "value": "FILTER"},
        
        "STABILITY_TYPE":
        {"comment": "EXPOSURE, GROUP, INSTRUMENT or FILE",
         "value": "INSTRUMENT"},
        
        "CENTROID_KEYS":
        {"comment": "Cat. parameters for centroiding",
         "value": "XWIN_IMAGE,YWIN_IMAGE"},
        
        "CENTROIDERR_KEYS":
        {"comment": 'Cat. params for centroid err ellipse',
         "value": "ERRAWIN_IMAGE,ERRBWIN_IMAGE,ERRTHETAWIN_IMAGE"},
        
        "DISTORT_KEYS":
        {"comment": 'Cat. parameters or FITS keywords',
         "value": "XWIN_IMAGE,YWIN_IMAGE"},
        
        "DISTORT_GROUPS":
        {"comment": "Filename for the check-image",
         "value": [1,1]},
        
        "DISTORT_DEGREES":
        {"comment": "Polynom degree for each group",
         "value": 3},
        
        "ASTREF_WEIGHT":
        {"comment": "Relative weight of ref.astrom.cat.",
         "value": 1.0},
        
        "ASTRCLIP_NSIGMA":
        {"comment": "Astrom. clipping threshold in sigmas",
         "value": 3.0},
        
        "CORRECT_COLOURSHIFTS":
        {"comment": 'Correct for colour shifts (Y/N)?',
         "value": "N"},

#---------------------------- Photometric solution ----------------------------
        "SOLVE_PHOTOM":
        {"comment": 'Compute photometric solution (Y/N) ?',
         "value": "Y"},
         
        "MAGZERO_OUT":
        {"comment": 'Magnitude zero-point(s) in output',
         "value": 0.0},
 
        "MAGZERO_INTERR":
        {"comment": 'Internal mag.zero-point accuracy',
         "value": 0.01},
         
        "MAGZERO_REFERR":
        {"comment": 'Photom.field mag.zero-point accuracy',
         "value": 0.03},

        "PHOTINSTRU_KEY":
        {"comment": 'FITS keyword(s) defining the photom.',
         "value": "FILTER"},

        "MAGZERO_KEY":
        {"comment": 'FITS keyword for the mag zero-point',
         "value": "PHOT_C"},

        "EXPOTIME_KEY":
        {"comment": 'FITS keyword for the exposure time (s)',
         "value": "EXPTIME"},

        "AIRMASS_KEY":
        {"comment": 'FITS keyword for the airmass',
         "value": "AIRMASS"},

        "EXTINCT_KEY":
        {"comment": 'FITS keyword for the extinction coeff',
         "value": "PHOT_K"},

        "PHOTOMFLAG_KEY":
        {"comment": 'FITS keyword for the photometry flag',
         "value": "PHOTFLAG"},
         
        "PHOTFLUX_KEY":
        {"comment": 'Catalog param. for the flux measurement',
         "value": "FLUX_AUTO"},         

        "PHOTFLUXERR_KEY":
        {"comment": 'Catalog parameter for the flux error',
         "value": "FLUXERR_AUTO"},
         
         "PHOTCLIP_NSIGMA":
        {"comment": 'Photom.clipping threshold in sigmas',
         "value": 3.0},
         
#------------------------------- Check-plots ----------------------------------
         
#         "CHECKPLOT_CKEY":
#        {"comment": 'FITS keyword for PLPLOT field colour',
#         "value": "SCAMPCOL"},
         
         "CHECKPLOT_DEV":
        {"comment": 'NULL, XWIN, TK, PS, PSC, XFIG, PNG, JPEG, AQT, PDF or SVG',
         "value": "PNG"},
         
#         "CHECKPLOT_RES":
#        {"comment": 'Check-plot resolution (0 = default)',
#         "value": 0},
         
#         "CHECKPLOT_ANTIALIAS":
#        {"comment": 'Anti-aliasing using convert (Y/N) ?',
#         "value": "Y"},
         
         "CHECKPLOT_TYPE":
        {"comment": 'Check-plots to perform',
         "value": "FGROUPS,DISTORTION,ASTR_INTERROR2D,ASTR_INTERROR1D,ASTR_REFERROR2D,ASTR_REFERROR1D,ASTR_CHI2,PHOT_ERROR"},
         
         "CHECKPLOT_NAME":
        {"comment": 'Check-plot filename(s)',
         "value": "fgroups,distort,astr_interror2d,astr_interror1d,astr_referror2d,astr_referror1d,astr_chi2,psphot_error"},
         
#------------------------------- Check-images ---------------------------------
         "CHECKIMAGE_TYPE":
        {"comment": 'NONE, AS_PAIR, AS_REFPAIR, or AS_XCORR',
         "value": "NONE"},
         
         "CHECKIMAGE_NAME":
        {"comment": 'Check-image filename(s)',
         "value": "check.fits"},
         

#------------------------------ Miscellaneous ---------------------------------

         "SN_THRESHOLDS":
        {"comment": 'S/N thresholds (in sigmas) for all and high-SN sample',
         "value": [5.0, 100.0]},
         
         "FWHM_THRESHOLDS":
        {"comment": 'FWHM thresholds (in pixels) for sources',
         "value": [2.0, 25.0]},
         
#         "FLAGS_MASK":
#        {"comment": 'Rejection mask on SEx FLAGS',
#         "value": "0x00f0"},
         
#         "WEIGHTFLAGS_MASK":
#        {"comment": 'Rejection mask on SEx FLAGS_WEIGHT',
#         "value": "0x00ff"},
         
#         "IMAFLAGS_MASK":
#        {"comment": 'Rejection mask on SEx IMAFLAGS_ISO',
#         "value": "0x0"},
         
#         "AHEADER_GLOBAL":
#        {"comment": 'Filename of the global INPUT header',
#         "value": "scamp.ahead"},
         
         "AHEADER_SUFFIX":
        {"comment": 'Filename extension for additional INPUT headers',
         "value": ".ahead"},
         
         "HEADER_SUFFIX":
        {"comment": 'Filename extension for OUTPUT headers',
         "value": ".head"},
         
         "HEADER_TYPE":
        {"comment": 'NORMAL or FOCAL_PLANE',
         "value": "NORMAL"},
         
         "VERBOSE_TYPE":
        {"comment": 'QUIET, NORMAL, LOG or FULL',
         "value": "FULL"},
         
         "WRITE_XML":
        {"comment": 'Write XML file (Y/N)?',
         "value": "Y"},
         
         "XML_NAME":
        {"comment": 'Filename for XML output',
         "value": "scamp.xml"},
         
         "XSL_URL":
        {"comment": 'Filename for XSL style-sheet',
         "value": "file:///usr/share/scamp/scamp.xsl"},
         
         "NTHREADS":
        {"comment": 'umber of simultaneous threads for the SMP version of SCAMP (0=automatic)',
         "value": 32},
         


        # -- Extra-keys (will not be saved in the main configuration file

        "CONFIG_FILE":
        {"comment": '[Extra key] name of the main configuration file',
         "value": "scamp.conf"}
    }
        
    
    # -- Special config. keys that should not go into the config. file.

    _SC_config_special_keys = ["CONFIG_FILE"]



    def __init__(self):
        """
        SCAMP class constructor.
        If a specific config_file is provided, it is used 
        """

        self.config = (
            dict([(k, copy.deepcopy(SCAMP._SC_config[k]["value"]))\
                  for k in SCAMP._SC_config.keys()]))

        # Extra config parameters that will be added/updated to the current values of the config file
        self.ext_config = {}
              
        self.program = None
        self.version = None


    def setup(self, path=None):
        """
        Look for SCAMP program ('scamp').
        If a full path is provided, only this path is checked.
        Raise a SCAMP_Exception if it failed.
        Return program and version if it succeed.
        """

        # -- Finding scamp program and its version
        # first look for 'scamp'

        candidates = ['scamp']

        if (path):
            candidates = [path]
        
        selected=None
        for candidate in candidates:
            try:
                p = subprocess.Popen (candidate, shell = True, bufsize = 0,
                    stdin = subprocess.PIPE, stdout = subprocess.PIPE, 
                    stderr = subprocess.STDOUT, close_fds = True)
                
                versionline = p.communicate()[0]
                #print(versionline.decode('ascii'))
 
                if (versionline.decode('ascii').find("SCAMP") != -1):
                    selected=candidate
                    break
            except (IOError, OSError):
                continue
                
        if not(selected):
            raise Exception("""
                  Cannot find SCAMP program. Check your PATH,
                  or provide the SCAMP program path in the constructor.
                  """)

        _program = selected

        #print(versionline)
        _version_match = re.search("[Vv]ersion ([0-9\.])+", versionline.decode('ascii'))
        if not _version_match:
            raise Exception("Cannot determine SCAMP version.")

        _version = _version_match.group()[8:]
        if not _version:
            raise Exception("Cannot determine SCAMP version.")

        # print("Use " + self.program + " [" + self.version + "]")

        return _program, _version

    def update_config(self):
        """
        Update the configuration files according to the current
        in-memory SCAMP configuration.
        """
        

        # -- Write main configuration file

        main_f = builtins.open(self.config['CONFIG_FILE'], 'w')

        for key in self.config.keys():
            if (key in SCAMP._SC_config_special_keys):
                continue

            if (key == "DISTORT_GROUPS" or key == "SN_THRESHOLDS" or key == "FWHM_THRESHOLDS" or key == "ASTREFMAG_LIMITS"): # tuple instead of a single value
                value = ",".join(map(str, self.config[key]))
            else:
                value = str(self.config[key])
            
            print("%-16s       %-16s # %s" %
                             (key, value, SCAMP._SC_config[key]['comment']), file=main_f)

        main_f.close()

    def run(self, catalog_list, updateconfig=True, clean=False, path=None):
        """
        Run SCAMP for a given list of catalog (.ldac files), and it can be one 
        single catalog list.

        Parameters
        ----------
        
        updateconfig: bool
            Is True (default), the configuration files will be updated before 
            running SCAMP.

        clean: bool
            If True (default: False), configuration files (if any) will be 
            deleted after SCAMP terminates.
        path: str
            Path name to look for scamp binary file in the system.
            
        Raises
        ------
        SCAMP_Exception
            Cannot run SCAMP
        SCAMP_AccuracyException
            SCAMP Warning/error: Significant inaccuracy likely to occur in 
            projection.
            
        """

        if updateconfig:
            self.update_config()

        # Try to find SCAMP program
        # This will raise an exception if it failed

        self.program, self.version = self.setup(path)
        
        # check how many files in the input
        my_catalogs=""
        if type(catalog_list) == list:
            for file in catalog_list:
               my_catalogs = my_catalogs + " " + file
        else:
            my_catalogs=catalog_list # a single file
        
            
        # Compound extra config command line args
        ext_args=""
        for key in self.ext_config.keys():
            ext_args=ext_args + " -" + key+ " " + str(self.ext_config[key])     
            
        commandline = (
            self.program + " -c " + self.config['CONFIG_FILE'] + " " + ext_args + " " + my_catalogs)
        
        #print(commandline)

        rcode = runCmd(commandline)
        
        if (rcode==1):
            #print("ERROR!!!")
            raise Exception("SCAMP command [%s] failed." % str(commandline))
        elif (rcode==2):
            raise Exception("SCAMP Warning/error: Significant inaccuracy likely to occur in projection.\n%s"%(commandline,))
            
        if clean:
            self.clean()

    def clean(self, config=True, catalog=False, check=False):
        """
        Remove the generated SCAMP files (if any).
        If config is True, remove generated configuration files.
        If catalog is True, remove the output catalog.
        If check is True, remove output check image.
        """

        try:
            if (config):
                os.unlink(self.config['CONFIG_FILE'])
            if (check):
                os.unlink(self.config['CHECKIMAGE_NAME'])
                
        except OSError:
            pass

        


In [ ]:
# HERE WE CREATE FRAME WITH WCS from solve-field run, 
# create the .ahead file for SCAMP,
# and run sextractor
#file_name = 'OMEGA.2014-03-10T01:39:59.047_pre.fits'
reduced_dir = '/home/fselman/omegacam/fernando/reduced/science/J1649-3307_164955p600_m330702p0/r_SDSS/2017-04-13T05:11:38/'
current_dir = '/home/fselman/omegacam/fernando/'
print('current directory:', os.getcwd())
os.chdir(reduced_dir)
print('current directory:', os.getcwd())
list_to_fix = glob.glob('OMEGA.*_pre')

if len(list_to_fix) > 0:
    i = 0
    for file_name in list_to_fix:
        i += 1
        if i <= len(list_to_fix):
            print('Fixing/sextracting %i/%i: %s'%(i, len(list_to_fix), file_name+'.fits',))
            create_wcs_frame(file_name+'.fits')
            write_scamp_ahead_file(file_name+'.fits')
            run_sextractor(file_name+'.fits')
                        
print('current directory:', os.getcwd())


In [ ]:
%%timeit

scamp = SCAMP()
# Using a specific config file (updateconfig=False)

scamp.config['ASTREF_CATALOG']="USNO-B1"

for file_name in list_to_fix:
    print(file_name+'_wcs.cat')
    
    os.chdir(file_name)
    # we remove a .head file from pervious runs if it exists
    old_scamp_cat_file = file_name+'_wcs.head'
    try:
        os.remove(old_scamp_cat_file)
    except OSError:
        pass
    
    scamp.config['CONFIG_FILE']="./scamp.conf"
    scamp.run([file_name+'_wcs.cat'], updateconfig=True, clean=False)
    os.chdir('..')

    